In [2]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.insert(0, str(Path.cwd().parent))  # points to src/
from shared_modeling import (
    compute_hassles_uplifts,
    compute_resilience_score,
    compute_stress_average,
    compute_stress_level,
    load_or_create_master_split_ids,
    run_model_experiment,
)

In [3]:
predictors = ['oDM', 'acog_PEgHTN', 'ChronHTN']  # diabetes, preeclampsia, chronic hypertension
df_health = pd.read_csv('../../Data/PREGNANCY_OUTCOMES.csv', usecols=predictors + ['PublicID'])
df_outcomes = pd.read_csv('../../Data/Modified/Outcome.csv', usecols=['PublicID', 'MH_outcome'])

# Mental health source datasets used as additional predictors in the maternal health models.
v2i_cols = ['PublicID'] + [f'V2IA{i:02d}' for i in range(1, 26)]
v1e_cols = ['PublicID', 'V1EA01', 'V1EA02a', 'V1EA02b', 'V1EA02c', 'V1EA02d', 'V1EA02e', 'V1EA02f', 'V1EA02g', 'V1EA02h', 'V1EA02i', 'V1EA02j', 'V1EA02k', 'V1EA02l']
v3j_cols = ['PublicID'] + [f'V3JA01{letter}' for letter in 'abcdefghij'] + [f'V3JA02{letter}' for letter in 'abcdefghij']
v1a_cols = ['PublicID', 'V1AH04', 'V1AH05', 'V1AH07', 'V1AH08']
v3a_cols = ['PublicID', 'V3AG04', 'V3AG05', 'V3AG07', 'V3AG08']
df_v2i = pd.read_csv('../../Data/V2I.csv', usecols=v2i_cols)
df_v1e = pd.read_csv('../../Data/V1E.CSV', usecols=v1e_cols, encoding='ISO-8859-1')
df_v3j = pd.read_csv('../../Data/V3J.csv', usecols=v3j_cols)
df_v1a = pd.read_csv('../../Data/V1A.CSV', usecols=v1a_cols)
df_v3a = pd.read_csv('../../Data/V3A.CSV', usecols=v3a_cols)

# Create the master split once and persist it for reuse in other notebooks.
split_path = 'master_split_ids.csv'
train_ids, test_ids = load_or_create_master_split_ids(df_outcomes, split_path)


In [4]:
df_v2i_features = compute_resilience_score(df_v2i)
df_v1e_features = compute_stress_average(df_v1e)
df_v3j_features = compute_hassles_uplifts(df_v3j)
df_v1a_v3a_features = compute_stress_level(pd.merge(df_v1a, df_v3a, on='PublicID', how='inner'))

mental_health_features = df_v2i_features.merge(df_v1e_features, on='PublicID', how='outer')
mental_health_features = mental_health_features.merge(df_v3j_features, on='PublicID', how='outer')
mental_health_features = mental_health_features.merge(df_v1a_v3a_features, on='PublicID', how='outer')


In [5]:
df = pd.merge(df_health, mental_health_features, on='PublicID', how='left')
df = pd.merge(df, df_outcomes, on='PublicID', how='inner')

feature_columns = [col for col in df.columns if col not in ['PublicID', 'MH_outcome']]
categorical_features = ['oDM', 'acog_PEgHTN']
ordinal_features = ['ResilienceLevel', 'StressLevel']
binary_features = ['ChronHTN']
numeric_features = [col for col in feature_columns if col not in categorical_features and col not in ordinal_features and col not in binary_features]
X = df[feature_columns]
y = df['MH_outcome']

train_df = df[df['PublicID'].isin(train_ids)].copy()
test_df = df[df['PublicID'].isin(test_ids)].copy()

X_train = train_df.drop(['MH_outcome', 'PublicID'], axis=1)
X_test = test_df.drop(['MH_outcome', 'PublicID'], axis=1)
y_train = train_df['MH_outcome']
y_test = test_df['MH_outcome']

df

,PublicID,oDM,ChronHTN,acog_PEgHTN,ResilienceTotalScore,ResilienceLevel,stress_average,FrequencyOfHassles,FrequencyOfUplifts,IntensityOfHassles,IntensityOfUplifts,HassleUpliftFrequencyRatio,HassleUpliftIntensityRatio,StressTotalScore,StressLevel,MH_outcome
0,00004O,3.0,2.0,7.0,102.0,1.0,1.846154,10.0,10.0,2.6,3.0,1.0,0.866667,24.0,0.5,1
1,00007I,3.0,2.0,7.0,106.0,1.0,1.538462,10.0,10.0,2.0,2.6,1.0,0.769231,18.0,0.5,1
2,00008G,3.0,2.0,7.0,107.0,1.0,2.076923,10.0,10.0,2.2,2.4,1.0,0.916667,17.0,0.5,0
3,00015J,3.0,2.0,7.0,120.0,1.0,1.692308,10.0,10.0,1.4,4.0,1.0,0.350000,10.0,0.0,0
4,00016H,3.0,2.0,7.0,97.0,2.0,2.000000,10.0,10.0,1.5,2.3,1.0,0.652174,18.0,0.5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7736,17349I,3.0,2.0,7.0,98.0,2.0,1.461538,10.0,10.0,1.7,2.4,1.0,0.708333,19.0,0.5,1
7737,17350A,3.0,2.0,3.0,119.0,1.0,1.384615,NaN,NaN,NaN,NaN,NaN,NaN,9.0,0.0,1
7738,17351V,3.0,2.0,7.0,104.0,1.0,1.384615,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
7739,17352T,3.0,2.0,7.0,100.0,2.0,1.538462,10.0,10.0,1.4,3.6,1.0,0.388889,17.0,0.5,1


In [5]:
# Run the LR pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'lr',
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    ordinal_features=ordinal_features,
    binary_features=binary_features
)


Dropping rows with missing values because impute=False (train: 495, test: 134).
Final dataset sizes for LR (impute=False): train=5697, test=1415
Fitting 5 folds for each of 30 candidates, totalling 150 fits


/Users/mkorpusi/Documents/GitHub/numom2b/nuMoM2b/src/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/mkorpusi/Documents/GitHub/numom2b/nuMoM2b/src/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/mkorpusi/Documents/GitHub/numom2b/nuMoM2b/src/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/mkorpusi/Documents/GitHub/numom2b/nuMoM2b/src/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/mkorpusi/Documents/GitHub/numom2b/nuMoM2b/src/.venv/lib/python3.13/site-packages/sklearn/linear_model

Best parameters found: {'classifier__C': 0.01, 'classifier__l1_ratio': 0.75}
Best CV Score (f1): 0.6827
Model Coefficients:
num__ResilienceTotalScore: -0.30742932733050476
num__stress_average: 0.45227480521276037
num__FrequencyOfHassles: 0.0
num__FrequencyOfUplifts: 0.0
num__IntensityOfHassles: 0.10371875264705113
num__IntensityOfUplifts: 0.0
num__HassleUpliftFrequencyRatio: 0.0
num__HassleUpliftIntensityRatio: 0.0335121634683502
num__StressTotalScore: 0.6518076748134088
ord__ResilienceLevel: 0.0
ord__StressLevel: 0.0
bin__ChronHTN: 0.0
cat__oDM_1.0: 0.0
cat__oDM_2.0: 0.0
cat__oDM_3.0: 0.0
cat__acog_PEgHTN_1.0: 0.0
cat__acog_PEgHTN_2.0: 0.0
cat__acog_PEgHTN_3.0: 0.0
cat__acog_PEgHTN_4.0: 0.0
cat__acog_PEgHTN_5.0: 0.0
cat__acog_PEgHTN_6.0: 0.0
cat__acog_PEgHTN_7.0: 0.0
Evaluation Metrics for LR with shared preprocessing and adaptive CV scoring:
Accuracy: 0.7187
Precision (positive class): 0.6383
Recall (positive class): 0.6965
F1 (positive class): 0.6661
Macro Precision: 0.7101
Macro Re

/Users/mkorpusi/Documents/GitHub/numom2b/nuMoM2b/src/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [7]:
# Run the RF pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'rf',
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    ordinal_features=ordinal_features,
    binary_features=binary_features
)


Dropping rows with missing values because impute=False (train: 495, test: 134).
Final dataset sizes for RF (impute=False): train=5697, test=1415
Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best parameters found: {'classifier__max_depth': 15, 'classifier__min_samples_leaf': 4, 'classifier__min_samples_split': 3, 'classifier__n_estimators': 700}
Best CV Score (f1): 0.6727
Feature Importances:
num__ResilienceTotalScore: 0.14347896171149838
num__stress_average: 0.17469853151480239
num__FrequencyOfHassles: 0.0001809834059045632
num__FrequencyOfUplifts: 0.0002895042118819352
num__IntensityOfHassles: 0.10245812359609098
num__IntensityOfUplifts: 0.07064830942623199
num__HassleUpliftFrequencyRatio: 0.0002996096184508755
num__HassleUpliftIntensityRatio: 0.10322653510057672
num__StressTotalScore: 0.24582801801698306
ord__ResilienceLevel: 0.04891780637748211
ord__StressLevel: 0.0696917021678575
bin__ChronHTN: 0.004581871190816208
cat__oDM_1.0: 0.0007495068141146382
cat__oDM_2.0: 

In [6]:
# Run the XGBoost pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'xgb',
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    ordinal_features=ordinal_features,
    binary_features=binary_features
)


Dropping rows with missing values because impute=False (train: 495, test: 134).
Final dataset sizes for XGB (impute=False): train=5697, test=1415
Fitting 5 folds for each of 243 candidates, totalling 1215 fits
Best parameters found: {'classifier__colsample_bytree': 0.8, 'classifier__learning_rate': 0.001, 'classifier__max_depth': 6, 'classifier__n_estimators': 60, 'classifier__subsample': 0.5}
Best CV Score (f1): 0.6766
Feature Importances:
num__ResilienceTotalScore: 0.0627143383026123
num__stress_average: 0.10721679776906967
num__FrequencyOfHassles: 0.0
num__FrequencyOfUplifts: 0.0
num__IntensityOfHassles: 0.03412993252277374
num__IntensityOfUplifts: 0.019155574962496758
num__HassleUpliftFrequencyRatio: 0.0
num__HassleUpliftIntensityRatio: 0.025458579882979393
num__StressTotalScore: 0.2712956368923187
ord__ResilienceLevel: 0.1264098733663559
ord__StressLevel: 0.19777552783489227
bin__ChronHTN: 0.02329389937222004
cat__oDM_1.0: 0.0
cat__oDM_2.0: 0.012305976822972298
cat__oDM_3.0: 0.016

In [7]:
# Run the SVM pipeline with macro F1 grid search.
best_model, y_pred, metrics = run_model_experiment(
    X_train,
    X_test,
    y_train,
    y_test,
    'svm',
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    ordinal_features=ordinal_features,
    binary_features=binary_features
)


Dropping rows with missing values because impute=False (train: 495, test: 134).
Final dataset sizes for SVM (impute=False): train=5697, test=1415
Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best parameters found: {'classifier__estimator__C': 0.1, 'classifier__estimator__gamma': 0.01, 'classifier__estimator__kernel': 'rbf'}
Best CV Score (f1): 0.6793
Skipping feature-level SVM output to keep notebook output compact.
Evaluation Metrics for SVM with shared preprocessing and adaptive CV scoring:
Accuracy: 0.7081
Precision (positive class): 0.6240
Recall (positive class): 0.6930
F1 (positive class): 0.6567
Macro Precision: 0.7001
Macro Recall: 0.7057
Macro F1: 0.7014
ROC AUC: 0.7810
